# KOZ-3 · v22 push, phase 2: experiment zoo (32 cores + RTX 4080 SUPER)

Runs in **`Codex Work/dinov3_wt`** (same as phase 1). Every experiment ends in the same honest gate as Codex's work:
nested honest > deployed, no outer fold regresses, k* ≥ 3, bootstrap P(gain>0) ≥ 0.8. Progress bars on every long step.

| # | experiment | resources | time |
|---|---|---|---|
| E1 | **hillfort localiser backbone zoo**: DINOv2-B, Satlas Swin-B (+ cached DINOv3-SAT) whole-scene tokens → probe ensembles → exact v22 test of the best combination | GPU (extraction) + CPU | ~30–60 min |
| E2 | **kurgan verifier zoo**: detector + context + image processing (LoG / DoG / contrast / symmetry / texture) + DINOv3 embeddings → LightGBM, CatBoost, XGBoost, NGBoost, AdaBoost, ExtraTrees, logistic, Optuna-tuned LightGBM, rank-mean ensemble → blended into the exact v22 decoding | all CPU cores | ~1–2 h |

Already known (phase 1 / earlier): DINOv3-SAT localiser **passes** (0.3797 → 0.4097); kurgan scene gate, CPU member,
member weights, LGB+Cat rescoring on detector features only **rejected**. On CPU here DINOv2-S alone hit 0.545 vs
DINOv3-SAT 0.691, so E1 probably confirms DINOv3-SAT; E2 is the open question (new image / embedding information).

Tip: E1 (GPU) and E2 (CPU) are independent; you can run them at the same time in two copies of this notebook.

In [ ]:
import os, subprocess
MAIN = "/home/nathan/Expedition Competition"; WT = os.path.join(MAIN, "Codex Work", "dinov3_wt"); os.chdir(WT)
FILES = ["scripts/sp_hillfort_dinov3_loc_oof.py", "scripts/p2_kurgan_verifier_zoo_oof.py", "scripts/sp_dinov3_verifier_oof.py",
         "scripts/robust_accept.py", "experiments/claude/p2", "experiments/claude/sp_cnn", "experiments/claude/sp_hillfort_dinov3_eval.py",
         "experiments/claude/hf_scene_cls.py", "experiments/claude/sp_dinov3_probe.py", "koz3/hf_dino_loc.py",
         "experiments/claude/outputs/hf_dinov3_loc_points_oof.csv"]
subprocess.run(["git", "fetch", "origin", "feat/physics-gates"], check=True)
subprocess.run(["git", "checkout", "origin/feat/physics-gates", "--"] + FILES, check=True)
for p in ["work/hf_dinov3/tokens.npz", "work/hf_dinov3/tokens768.npz", "work/spor/oof_scene_cls.parquet", "work/spor/oof_dinov3.npz",
          "kaggle_import/code/docs/site_crs.csv"]:
    print(("OK      " if os.path.exists(p) else "MISSING ") + p)
os.makedirs("logs", exist_ok=True)

## E1 · hillfort localiser backbone zoo
### E1a · whole-scene tokens on the GPU (progress bar per backbone)

In [ ]:
!python experiments/claude/p2/backbone_tokens.py dinov2_b 518 work/hf_dinov3/tok_dinov2b_518.npz
!python experiments/claude/p2/backbone_tokens.py dinov2_b 770 work/hf_dinov3/tok_dinov2b_770.npz
!python experiments/claude/p2/backbone_tokens.py satlas 512 work/hf_dinov3/tok_satlas_512.npz
!python experiments/claude/p2/backbone_tokens.py satlas 768 work/hf_dinov3/tok_satlas_768.npz

### E1b · out-of-fold hit rate (point inside the hillfort) for each backbone and ensembles; current = 0.691

In [ ]:
T = "work/hf_dinov3/"
combos = {"dinov3 (current)": ["tokens.npz", "tokens768.npz"],
          "dinov2_b": ["tok_dinov2b_518.npz", "tok_dinov2b_770.npz"],
          "satlas": ["tok_satlas_512.npz", "tok_satlas_768.npz"],
          "dinov3+satlas": ["tok_satlas_512.npz", "tok_satlas_768.npz", "tokens.npz", "tokens768.npz"],
          "dinov3+dinov2_b": ["tok_dinov2b_518.npz", "tok_dinov2b_770.npz", "tokens.npz", "tokens768.npz"],
          "all": ["tok_dinov2b_518.npz", "tok_dinov2b_770.npz", "tok_satlas_512.npz", "tok_satlas_768.npz", "tokens.npz", "tokens768.npz"]}
from tqdm.auto import tqdm
res = {}
for name, files in tqdm(combos.items(), desc="localiser combos"):
    out = f"work/hf_dinov3/loc_{name.replace(' ', '_').replace('(', '').replace(')', '').replace('+', '_')}.csv"
    r = subprocess.run(["python", "experiments/claude/sp_hillfort_dinov3_eval.py"] + [T + f for f in files],
                       capture_output=True, text=True, env={**os.environ, "LOC_OUT": out})
    lines = [l for l in r.stdout.splitlines() if l.startswith("[") or l.startswith("saved")]
    res[name] = (out, lines); print(f"--- {name}"); print("\n".join(lines[-2:]) or r.stderr[-800:])

### E1c · exact v22 satellite test for the ensembles (only worth it if a combo's hit beats 0.691 clearly)
Each run takes ~30–60 min; it scores the new points with the same settings as phase-1 test A (gate unchanged).

In [ ]:
import pandas as pd
cur = 0.691
for name, (out, _) in res.items():
    if name == "dinov3 (current)" or not os.path.exists(out): continue
    D = pd.read_csv(out); hit = D[D.has_hillfort].hit.mean()
    print(f"{name:18s} fixed-ensemble hit {hit:.3f}", "→ exact test" if hit >= cur + 0.02 else "(not better: skip)")
    if hit >= cur + 0.02:
        log = f"logs/claude_E1_{name.replace('+', '_')}.log"
        !LOC_POINTS={out} python scripts/sp_hillfort_dinov3_loc_oof.py 2>&1 | grep -v -i "warn\|concat" | tee {log} | tail -6

## E2 · kurgan verifier zoo (all CPU cores; Optuna progress bars)
Writes image features once (`work/spor/p2_img_feats.parquet`), then trains the zoo per outer fold and scores every
(model, λ) in the exact decoding. Use `--quick` first (fewer Optuna trials, no NGBoost) if you want a fast look.

In [ ]:
!HF_HUB_OFFLINE=1 python scripts/p2_kurgan_verifier_zoo_oof.py 2>&1 | grep -v -i "warn\|concat" | tee logs/claude_E2_verifier_zoo.log | tail -70

## Results to paste back

In [ ]:
import glob
for p in sorted(glob.glob("logs/claude_E*.log")):
    print("=" * 30, p); print(open(p).read()[-7000:])
for name, (out, lines) in (res.items() if 'res' in globals() else []):
    print(name, lines[-2:] if lines else "")